In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# Convert a number to hex and return a list of digit tokens
def number_to_hex_digits(num):
    hex_str = hex(num)[2:].upper()  # Convert to hex (remove '0x' and make uppercase)
    return [int(char, 16) for char in hex_str]  # Convert hex characters to indices

# Generate dataset: multiple numbers → hex → sorted hex list
def generate_data(num_samples=1000, max_num=999):
    data = []
    for _ in range(num_samples):
        numbers = [random.randint(0, max_num) for _ in range(random.randint(1, 5))]  # Random list of 1-5 numbers
        hex_digits = [digit for num in numbers for digit in number_to_hex_digits(num)]  # Flatten hex digits
        sorted_hex_digits = sorted(hex_digits)  # Sorting step
        data.append((hex_digits, sorted_hex_digits))
    return data

# Token and Positional Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

    def forward(self, x):
        return self.embedding(x)

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=20):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        return x + self.pos_embedding(positions)

# Transformer Encoder Layer
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_output)
        ffn_output = self.ffn(x)
        return self.norm2(x + ffn_output)

# Transformer Encoder Model
class TransformerSorter(nn.Module):
    def __init__(self, vocab_size=16, embed_dim=64, num_heads=4, hidden_dim=128, num_layers=4):
        super().__init__()
        self.token_embed = TokenEmbedding(vocab_size, embed_dim)
        self.positional_encoding = PositionalEncoding(embed_dim)
        self.encoder_layers = nn.Sequential(*[TransformerEncoderLayer(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)])
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.token_embed(x)
        x = self.positional_encoding(x)
        x = self.encoder_layers(x)
        return self.output_layer(x)

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 66
LR = 0.0001

# Model, Loss, Optimizer
model = TransformerSorter().to(device)  # Move model to GPU if available
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)

# Generate Training Data
dataset = generate_data()
train_data = [(torch.tensor(x, device=device), torch.tensor(y, device=device)) for x, y in dataset]

scaler = torch.cuda.amp.GradScaler()

for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in train_data:
        x, y = x.unsqueeze(0), y.unsqueeze(0)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():  # Mixed precision
            output = model(x)
            loss = criterion(output.view(-1, 16), y.view(-1))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_data):.4f}")

# Test Prediction Function
def predict_sorting(model, numbers):
    model.eval()
    with torch.no_grad():
        hex_seq = [digit for num in numbers for digit in number_to_hex_digits(num)]
        x = torch.tensor(hex_seq, device=device).unsqueeze(0)  # Add batch dim
        output = model(x)
        predicted = torch.argmax(output, dim=-1).squeeze(0)
        predicted_hex = [format(p.item(), 'X') for p in predicted]
        return predicted_hex



Training on: cuda


<ipython-input-3-d03112edd861>:91: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
<ipython-input-3-d03112edd861>:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():  # Mixed precision


Epoch 1, Loss: 2.1354
Epoch 2, Loss: 1.4279
Epoch 3, Loss: 1.0062
Epoch 4, Loss: 0.7344
Epoch 5, Loss: 0.5607
Epoch 6, Loss: 0.4372
Epoch 7, Loss: 0.3434
Epoch 8, Loss: 0.2738
Epoch 9, Loss: 0.2191
Epoch 10, Loss: 0.2108
Epoch 11, Loss: 0.1538
Epoch 12, Loss: 0.1252
Epoch 13, Loss: 0.1137
Epoch 14, Loss: 0.1060
Epoch 15, Loss: 0.0928
Epoch 16, Loss: 0.0940
Epoch 17, Loss: 0.0675
Epoch 18, Loss: 0.0761
Epoch 19, Loss: 0.0607
Epoch 20, Loss: 0.0380
Epoch 21, Loss: 0.0633
Epoch 22, Loss: 0.0496
Epoch 23, Loss: 0.0251
Epoch 24, Loss: 0.0610
Epoch 25, Loss: 0.0440
Epoch 26, Loss: 0.0325
Epoch 27, Loss: 0.0260
Epoch 28, Loss: 0.0351
Epoch 29, Loss: 0.0412
Epoch 30, Loss: 0.0173
Epoch 31, Loss: 0.0093
Epoch 32, Loss: 0.0850
Epoch 33, Loss: 0.0303
Epoch 34, Loss: 0.0110
Epoch 35, Loss: 0.0394
Epoch 36, Loss: 0.0200
Epoch 37, Loss: 0.0344
Epoch 38, Loss: 0.0239
Epoch 39, Loss: 0.0187
Epoch 40, Loss: 0.0262
Epoch 41, Loss: 0.0338
Epoch 42, Loss: 0.0123
Epoch 43, Loss: 0.0238
Epoch 44, Loss: 0.06

In [ ]:
numbers = [25, 8, 250]  # Example list of numbers
sorted_hex = predict_sorting(model, numbers)
print(f"Original Numbers: {numbers} → Sorted Hex Digits: {sorted_hex}")

Original Numbers: [25, 8, 250] → Sorted Hex Digits: ['1', '8', '9', 'A', 'F']


In [ ]:
torch.save(model, 'full_model.pth')

In [ ]:
# Save the model's state_dict instead of the full model
torch.save(model.state_dict(), 'model_state_dict.pth')


Actual Code Starts from below

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# Convert a number to hex and return a list of digit tokens
def number_to_hex_digits(num):
    hex_str = hex(num)[2:].upper()  # Convert to hex (remove '0x' and make uppercase)
    return [int(char, 16) for char in hex_str]  # Convert hex characters to indices

# Generate dataset: multiple numbers → hex → sorted hex list
def generate_data(num_samples=2000, max_num=999):
    data = []
    for _ in range(num_samples):
        numbers = [random.randint(0, max_num) for _ in range(random.randint(1, 5))]  # Random list of 1-5 numbers
        hex_digits = [digit for num in numbers for digit in number_to_hex_digits(num)]  # Flatten hex digits
        sorted_hex_digits = sorted(hex_digits)  # Sorting step
        data.append((hex_digits, sorted_hex_digits))
    return data

# Token and Positional Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

    def forward(self, x):
        return self.embedding(x)

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=20):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        return x + self.pos_embedding(positions)

# Transformer Encoder Layer
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_output)
        ffn_output = self.ffn(x)
        return self.norm2(x + ffn_output)

# Transformer Encoder Model
class TransformerSorter(nn.Module):
    def __init__(self, vocab_size=16, embed_dim=128, num_heads=16, hidden_dim=256, num_layers=6):
        super().__init__()
        self.token_embed = TokenEmbedding(vocab_size, embed_dim)
        self.positional_encoding = PositionalEncoding(embed_dim)
        self.encoder_layers = nn.Sequential(*[TransformerEncoderLayer(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)])
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.token_embed(x)
        x = self.positional_encoding(x)
        x = self.encoder_layers(x)
        return self.output_layer(x)

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 50
LR = 0.0005
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.9)
# Model, Loss, Optimizer
model = TransformerSorter().to(device)  # Move model to GPU if available
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)

# Generate Training Data
dataset = generate_data()
train_data = [(torch.tensor(x, device=device), torch.tensor(y, device=device)) for x, y in dataset]

scaler = torch.amp.GradScaler()

for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in train_data:
        x, y = x.unsqueeze(0), y.unsqueeze(0)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            output = model(x)
            loss = criterion(output.view(-1, 16), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_data):.4f}")

# Test Prediction Function
def predict_sorting(model, numbers):
    model.eval()
    with torch.no_grad():
        with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            hex_seq = [digit for num in numbers for digit in number_to_hex_digits(num)]
            x = torch.tensor(hex_seq, device=device).unsqueeze(0)  # Add batch dim
            output = model(x)
            predicted = torch.argmax(output, dim=-1).squeeze(0)
            predicted_hex = [format(p.item(), 'X') for p in predicted]
            return predicted_hex

Training on: cuda
Epoch 1, Loss: 1.0667
Epoch 2, Loss: 0.5729
Epoch 3, Loss: 0.4253
Epoch 4, Loss: 0.3558
Epoch 5, Loss: 0.3153
Epoch 6, Loss: 0.2770
Epoch 7, Loss: 0.2565
Epoch 8, Loss: 0.2191
Epoch 9, Loss: 0.2294
Epoch 10, Loss: 0.1962
Epoch 11, Loss: 0.1946
Epoch 12, Loss: 0.1750
Epoch 13, Loss: 0.1967
Epoch 14, Loss: 0.1501
Epoch 15, Loss: 0.1518
Epoch 16, Loss: 0.1348
Epoch 17, Loss: 0.1402
Epoch 18, Loss: 0.1246
Epoch 19, Loss: 0.1220
Epoch 20, Loss: 0.1257
Epoch 21, Loss: 0.1245
Epoch 22, Loss: 0.1080
Epoch 23, Loss: 0.1094
Epoch 24, Loss: 0.0938
Epoch 25, Loss: 0.1045
Epoch 26, Loss: 0.0914
Epoch 27, Loss: 0.0880
Epoch 28, Loss: 0.0841
Epoch 29, Loss: 0.0854
Epoch 30, Loss: 0.1018
Epoch 31, Loss: 0.0781
Epoch 32, Loss: 0.0858
Epoch 33, Loss: 0.0680
Epoch 34, Loss: 0.0806
Epoch 35, Loss: 0.0662
Epoch 36, Loss: 0.0827
Epoch 37, Loss: 0.0622
Epoch 38, Loss: 0.0700
Epoch 39, Loss: 0.0639
Epoch 40, Loss: 0.0679
Epoch 41, Loss: 0.0805
Epoch 42, Loss: 0.0544
Epoch 43, Loss: 0.0602
Ep

In [ ]:
numbers = [25, 8, 160, 60, 45]  # Example list of numbers
sorted_hex = predict_sorting(model, numbers)
print(f"Original Numbers: {numbers} → Sorted Hex Digits: {sorted_hex}")

Original Numbers: [25, 8, 160, 60, 45] → Sorted Hex Digits: ['0', '1', '2', '3', '8', '9', 'A', 'C', 'D']


In [ ]:
torch.save(model, 'full_model.pth')

In [ ]:
# Save the model's state_dict instead of the full model
torch.save(model.state_dict(), 'model_state_dict.pth')

below i am loading using the model (after training i am coming back to test and model)

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Convert a number to hex and return a list of digit tokens
def number_to_hex_digits(num):
    hex_str = hex(num)[2:].upper()  # Convert to hex (remove '0x' and make uppercase)
    return [int(char, 16) for char in hex_str]  # Convert hex characters to indices

# Generate dataset: multiple numbers → hex → sorted hex list
def generate_data(num_samples=2000, max_num=999):
    data = []
    for _ in range(num_samples):
        numbers = [random.randint(0, max_num) for _ in range(random.randint(1, 5))]  # Random list of 1-5 numbers
        hex_digits = [digit for num in numbers for digit in number_to_hex_digits(num)]  # Flatten hex digits
        sorted_hex_digits = sorted(hex_digits)  # Sorting step
        data.append((hex_digits, sorted_hex_digits))
    return data

# Token and Positional Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

    def forward(self, x):
        return self.embedding(x)

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=20):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        return x + self.pos_embedding(positions)

# Transformer Encoder Layer
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_output)
        ffn_output = self.ffn(x)
        return self.norm2(x + ffn_output)

# Transformer Encoder Model
class TransformerSorter(nn.Module):
    def __init__(self, vocab_size=16, embed_dim=128, num_heads=16, hidden_dim=256, num_layers=6):
        super().__init__()
        self.token_embed = TokenEmbedding(vocab_size, embed_dim)
        self.positional_encoding = PositionalEncoding(embed_dim)
        self.encoder_layers = nn.Sequential(*[TransformerEncoderLayer(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)])
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.token_embed(x)
        x = self.positional_encoding(x)
        x = self.encoder_layers(x)
        return self.output_layer(x)

# Load the saved model
model = torch.load('retrained_model.pth', map_location=device, weights_only=False)
model.eval()  # Set the model to evaluation mode

# Test Prediction Function
def predict_sorting(model, numbers):
    with torch.no_grad():
        hex_seq = [digit for num in numbers for digit in number_to_hex_digits(num)]
        x = torch.tensor(hex_seq, device=device).unsqueeze(0)  # Add batch dim
        output = model(x)
        predicted = torch.argmax(output, dim=-1).squeeze(0)
        predicted_hex = [format(p.item(), 'X') for p in predicted]
        return predicted_hex

# Example usage: Predict sorted hex digits from numbers
numbers =    [13, 50, 300, 77, 19, 150] # Example list of numbers
sorted_hex = predict_sorting(model, numbers)
print(f"Original Numbers: {numbers} → Sorted Hex Digits: {sorted_hex}")


Using device: cuda
Original Numbers: [13, 50, 300, 77, 19, 150] → Sorted Hex Digits: ['1', '1', '2', '2', '3', '3', '4', '6', '9', 'C', 'D', 'D']


Below i am training the model

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Convert a number to hex and return a list of digit tokens
def number_to_hex_digits(num):
    hex_str = hex(num)[2:].upper()  # Convert to hex (remove '0x' and make uppercase)
    return [int(char, 16) for char in hex_str]  # Convert hex characters to indices

def generate_data(num_samples=5000, max_num=999):
    data = []
    for _ in range(num_samples):
        numbers = [random.randint(0, max_num) for _ in range(random.randint(1, 5))]  # Random list of 1-5 numbers
        hex_digits = [digit for num in numbers for digit in number_to_hex_digits(num)]  # Flatten hex digits
        sorted_hex_digits = sorted(hex_digits)  # Sorting step
        data.append((hex_digits, sorted_hex_digits))
    return data
# Define your classes again (TokenEmbedding, PositionalEncoding, TransformerEncoderLayer, TransformerSorter)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

    def forward(self, x):
        return self.embedding(x)


class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=20):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        return x + self.pos_embedding(positions)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_output)
        ffn_output = self.ffn(x)
        return self.norm2(x + ffn_output)

class TransformerSorter(nn.Module):
    def __init__(self, vocab_size=16, embed_dim=128, num_heads=16, hidden_dim=256, num_layers=6):
        super().__init__()
        self.token_embed = TokenEmbedding(vocab_size, embed_dim)
        self.positional_encoding = PositionalEncoding(embed_dim)
        self.encoder_layers = nn.Sequential(*[TransformerEncoderLayer(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)])
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.token_embed(x)
        x = self.positional_encoding(x)
        x = self.encoder_layers(x)
        return self.output_layer(x)

# Add the custom class to the safe globals
torch.serialization.add_safe_globals([TransformerSorter])

# Load the saved model
model = torch.load('retrained_model.pth', map_location=device, weights_only=False)
model.to(device)  # Ensure model is on the right device
model.train()  # Set the model to training mode

# Hyperparameters
BATCH_SIZE = 64
EPOCHS = 5  # Retraining for only 10 epochs
LR = 0.0001
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.9)

# Loss function
criterion = nn.CrossEntropyLoss()

# Generate Training Data
dataset = generate_data()
train_data = [(torch.tensor(x, device=device), torch.tensor(y, device=device)) for x, y in dataset]

# Training loop for 10 epochs
for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in train_data:
        x, y = x.unsqueeze(0), y.unsqueeze(0)  # Add batch dimension

        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output.view(-1, 16), y.view(-1))  # CrossEntropy expects 2D output (batch_size * num_classes)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_data):.4f}")

# Save the retrained model (optional)
torch.save(model, 'retrained_model.pth')

# Test Prediction Function
def predict_sorting(model, numbers):
    with torch.no_grad():
        hex_seq = [digit for num in numbers for digit in number_to_hex_digits(num)]
        x = torch.tensor(hex_seq, device=device).unsqueeze(0)  # Add batch dim
        output = model(x)
        predicted = torch.argmax(output, dim=-1).squeeze(0)
        predicted_hex = [format(p.item(), 'X') for p in predicted]
        return predicted_hex

# Example usage: Predict sorted hex digits from numbers
numbers = [25, 8, 160, 60, 45]  # Example list of numbers
sorted_hex = predict_sorting(model, numbers)
print(f"Original Numbers: {numbers} → Sorted Hex Digits: {sorted_hex}")


Using device: cuda
Epoch 1, Loss: 0.0022
Epoch 2, Loss: 0.0010
Epoch 3, Loss: 0.0005
Epoch 4, Loss: 0.0008
Epoch 5, Loss: 0.0010
Original Numbers: [25, 8, 160, 60, 45] → Sorted Hex Digits: ['0', '1', '2', '3', '8', '9', 'A', 'C', 'D']
